In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'Baseline_Training', 'CTC_models', 'PhoneticFeatures_Training'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    BASELINE_MODELS_DIR,
    CTC_MODELS_DIR,
    DEFAULT_BASELINE_MODEL,
    DEFAULT_CTC_ATTENTION_MODEL,
    DEFAULT_CTC_MODEL,
    DEFAULT_PHONETIC_MODEL_2CLASS,
    DEFAULT_PHONETIC_MODEL_3CLASS,
    EXAMPLE_EMBEDDINGS,
    EXAMPLE_PHONEMES,
    EXAMPLE_SEG_B2,
    EXAMPLE_SEG_B4,
    EXAMPLE_WAV,
    PHONEME_CHOICES_SUMMARY,
    PHONETIC_MODELS_DIR,
    TABLES_DIR,
)


In [ ]:
ищimport torch
from DatasetsModels import PhonemeEmbeddingDataset, ClassificationModel, collate_fn, ClassificationModelCTChead
from GetData import load_selected_embeddings
from torch.utils.data import DataLoader
from VizulalisationTools import chosen_phonemes_stats
from main import train, test, evaluate
import random
from SaveTools import save_experiment_info
import os
from torch.utils.tensorboard import SummaryWriter

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA devices: {torch.cuda.device_count()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA runtime version: {torch.version.cuda}")
print(f"cuDNN version: {torch.backends.cudnn.version() if torch.cuda.is_available() else 'N/A'}")

# Проверь установку PyTorch
print(f"torch.cuda.is_built(): {torch.backends.cudnn.enabled}")



In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

# Данные

Пути

In [ ]:
root_dir = r'\\nid-tts-02\mnt\hot_store\trainee_common\ananeva\CORPRES_embs_no_averaging'
save_path = str(PROJECT_ROOT)
speakers = ['Vladimir', 'Maria', 'Mikhail', 'Anna', 'Galina', 'Victoria', 'Petr', 'Alexander']
phoneme_list_full = ['a0', 'a1', 'a2', 'a4', 'b', "b'", 'c', 'ch', 'ch_', 'd', "d'", 'e0', 'e1', 'e2', 'e4', 'f', "f'", 'g', "g'", 'h', "h'", 'i0', 'i1', 'i2', 'i4', 'j', 'jr', 'jl', 'ji4', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'o1', 'o2', 'o4', 'p', "p'", 'r', "r'", 's', "s'", 'sc', 'sh', 't', "t'", 'u0', 'u1', 'u2', 'u4', 'v', "v'", 'y0', 'y1', 'y2', 'y4', 'z', "z'", 'zh', "zh'", 'C', 'CH', 'H', 'SC']

In [ ]:
debugging = False
save_exp = True
training = True
#model_id = 'VACXUXVEXO'
training_mode = 'triplet' # ['triplet', 'ctc', 'averaged']
phoneme_list = ['a0', 'a4', 'b', "b'", 'c', 'ch', 'd', "d'", 'e0', 'f', "f'", 'g', 'h', 'i0','i4', 'j', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'p', "p'", 'r', "r'", 's', "s'", 'sh', 't', "t'", 'u0', 'v', "v'", 'y0', 'z', "z'", 'zh', 'sil', 'a1', 'i1','u1', 'y1', ]
# phoneme_list = ['a0', 'a4', 'b', 'd','e0', 'f',  'g', 'h', 'i0','i4', 'j', 'k','l', 'm',  'n', 'o0', 'p', 'r', 's', 'sh', 't', 'u0', 'v',  'y0', 'z', 'zh', 'sil']
# phoneme_list = ['a0', 't', "i0", "t'", "k", 's', "s'" ]  
# интересующие фонемы (без позиций)
samples_of_ph = 1000              # сколько примеров на фонему
train_speakers = ['Vladimir', 'Maria', 'Mikhail', 'Anna']
test_speakers = ['Galina', 'Victoria', 'Petr', 'Alexander']

if debugging:
    phoneme_list = ['a0', 't', "i0" ]  
    samples_of_ph = 1            
    train_speakers = ['Vladimir']
    test_speakers = ['Galina']
    

In [ ]:
int_code = [random.randint(65, 90) for i in range(10)]
uid = ''.join([chr(code) for code in int_code])

In [ ]:
log_path = save_path + '\\runs\\' + uid

In [ ]:
log_path

In [ ]:
num_classes = len(phoneme_list)

In [ ]:
learning_rate = 1e-3
weight_decay=1e-4
batch_size = 32
num_epochs = 20
dropout = 0.1

# Обучение

## Данные

In [ ]:
train_embeddings_border, train_phonemes_border, _ = load_selected_embeddings(root_dir, phoneme_list, samples_of_ph, train_speakers,  triplet_variation='one border')
train_embeddings_nobord, train_phonemes_nobord, _ = load_selected_embeddings(root_dir, phoneme_list, 500, train_speakers,  triplet_variation='no borders')

test_embeddings_border, test_phonemes_border, _ = load_selected_embeddings(root_dir, phoneme_list, samples_of_ph, test_speakers,  triplet_variation='one border')
test_embeddings_nobord, test_phonemes_nobord, _ = load_selected_embeddings(root_dir, phoneme_list, 500, test_speakers,  triplet_variation='no borders')


In [ ]:
test_embeddings = test_embeddings_border + test_embeddings_nobord
test_phonemes = test_phonemes_border + test_phonemes_nobord

train_phonemes = train_phonemes_border + train_phonemes_nobord
train_embeddings = train_embeddings_border + train_embeddings_nobord

In [ ]:
train_dataset = PhonemeEmbeddingDataset(train_embeddings, phoneme_list)
test_dataset = PhonemeEmbeddingDataset(test_embeddings, phoneme_list)

In [ ]:
emb_shape = 256
for data in train_dataset:
    embb, number_label, label = data
    if embb.shape != torch.Size([256]) or number_label.shape != torch.Size([3]) or len(label)!=3 :
        print(embb.shape, number_label.shape, len(label))
        print(label)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collate_fn)# sampler=sampler)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [ ]:
input_size = len(train_embeddings[0]['embedding'])

In [ ]:
stats, tr_stats = chosen_phonemes_stats(train_embeddings)
        

## Модель

In [ ]:
training_mode = 'triplets'
if training_mode == 'triplets':
    model = ClassificationModel(input_size, num_classes, dropout=dropout).to(device)
    loss_fn = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
if training_mode == 'ctc':
    blank = 'blank'
    char2id = {'blank':0}
    id2char = {0:blank}
    for i, ch in  enumerate(phoneme_list):
        char2id[ch] = i
        id2char[i] = ch
    def text_to_id(text):
        return [char2id[ch] for ch in text]
    
    model = ClassificationModelCTChead(input_size, num_classes, dropout=dropout).to(device)
    loss_fn = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

In [ ]:
device

In [ ]:
if training:
    writer = SummaryWriter(log_dir=log_path)
    print(log_path)
    print('training')
    train(model, train_loader, test_loader, loss_fn, optimizer, scheduler, 10, device, phoneme_list, writer=writer,loss_info=True)
    writer.close()
else:
    model = torch.load(os.path.join(save_path, model_id + '_model.pth'))

In [ ]:
result = test(model, test_loader, loss_fn, uid, phoneme_list=phoneme_list, device=device)

In [ ]:
if save_exp:
    params = {
        'uid': uid,
        'lr': learning_rate,
        'weight_decay': weight_decay,
        'batch_size': batch_size,
        'num_epochs': num_epochs,
        'dropout': dropout,
    }
    data = {
        "phonemes": samples_of_ph,
        "number" : samples_of_ph,
        'speakers': [train_speakers, test_speakers],
    }
    metrics = evaluate(model, test_loader, loss_fn, uid, phoneme_list, device=device, show_plot=False)
    save_experiment_info(save_path, params, data, metrics, uid)

In [ ]:
if training and save_exp:
    name = uid + '_model.pth'
    model_save_path = (os.path.join(save_path, name))
    torch.save(model, model_save_path)
    print("model saved", model_save_path)

## Decoding Evaluation

In [ ]:
from collections import Counter
import numpy as np
from GetData import load_phrases

In [ ]:
embedding_sequences, phoneme_targets, info = load_phrases(
    root_dir=root_dir,
    speakers=['Galina', 'Victoria', 'Petr', 'Alexander'],
    max_sequences_per_speaker=1000)

In [ ]:
from Decoding_tools import get_phrase

for l in phoneme_targets:
    print(list(l.keys())[0])
    print(get_phrase(list(l.values())[0]))

In [ ]:
prob_distributions = {'start': [], 'centre': [], 'end': []}
y_pred = {'start': [], 'centre': [], 'end': []}
result = []
phoneme_list = ['a0', 'a4', 'b', "b'", 'c', 'ch', 'd', "d'", 'e0', 'f', "f'", 'g', 'h', 'i0','i4', 'j', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'p', "p'", 'r', "r'", 's', "s'", 'sh', 't', "t'", 'u0', 'v', "v'", 'y0', 'z', "z'", 'zh', 'sil']
from main import get_phrase_prediciton
from evaluation_tools import compute_error_rate
from tqdm import tqdm

In [ ]:
overall_cer_la = 0
overall_cer = 0
for phrase, targets in tqdm(zip(embedding_sequences, phoneme_targets)):
    y_pred,prob_distributions, result = get_phrase_prediciton(phrase, model, phoneme_list, device)
    true_phrase, true_list = get_phrase(list(targets.values())[0])
    metrics = compute_error_rate(prob_distributions, true_list)
    overall_cer_la += metrics['metrics']['la_cer']
    overall_cer += metrics['metrics']['basic_cer']


In [ ]:
print(overall_cer_la/len(phoneme_targets))
print(overall_cer/len(phoneme_targets))